<a href="https://colab.research.google.com/github/dejairps85-afk/ai-trading-agent/blob/feature%2Fwin-wdo-trustworthy-ingestion/notebooks/AI_Trading_Agent_Research_Lab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# AI Trading Agent — Research Lab
### WIN / WDO · B3 · PAPER_ONLY

Este notebook prepara um ambiente reproduzível para importar, auditar e explorar dados históricos de WIN e WDO.

**Controles permanentes deste notebook:**
- Não conecta a corretoras nem à ProfitDLL.
- Não envia, altera ou cancela ordens.
- Não presume que candles OHLCV sejam negócios individuais.
- Não inventa agressão compradora/vendedora nem livro de ofertas.
- Não treina modelos automaticamente.
- Não considera a origem dos dados autenticada apenas porque o CSV foi carregado.

**Como usar no iPhone:** abra este arquivo no Google Colab, execute as células de cima para baixo e, quando solicitado, carregue seus arquivos CSV ou ZIP.

## 1. Inicialização e controles de segurança

A primeira célula instala somente bibliotecas comuns de análise de dados e cria a estrutura de diretórios da sessão.

In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import os, sys, json, zipfile, hashlib, shutil, warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

MODE = "PAPER_ONLY"
LIVE_TRADING = False
ORDER_ROUTING = False
BROKER_CONNECTION = False

assert MODE == "PAPER_ONLY"
assert LIVE_TRADING is False
assert ORDER_ROUTING is False
assert BROKER_CONNECTION is False

SEED = 42
np.random.seed(SEED)

BASE = Path("/content/ai_trading_agent")
RAW = BASE / "data" / "raw"
AUDITED = BASE / "data" / "audited"
PROCESSED = BASE / "data" / "processed"
REPORTS = BASE / "reports"

for folder in (RAW, AUDITED, PROCESSED, REPORTS):
    folder.mkdir(parents=True, exist_ok=True)

print("AI Trading Agent — Research Lab")
print("Modo:", MODE)
print("Execução real:", LIVE_TRADING)
print("Roteamento:", ORDER_ROUTING)
print("Python:", sys.version.split()[0])
print("Pandas:", pd.__version__)
print("NumPy:", np.__version__)
print("UTC:", datetime.now(timezone.utc).isoformat())
print("Diretório:", BASE)

AI Trading Agent — Research Lab
Modo: PAPER_ONLY
Execução real: False
Roteamento: False
Python: 3.13.18
Pandas: 2.2.3
NumPy: 2.4.6
UTC: 2026-11-27T14:30:52.698268+00:00
Diretório: /content/ai_trading_agent


## 2. Carregar arquivos

Carregue CSVs ou ZIPs que você tenha direito de utilizar. O conteúdo será copiado para `data/raw`.

**Nota sobre os arquivos já encontrados na pesquisa:** os CSVs M15 de WIN/WDO do repositório `geansm2/quant-research` não apresentam licença de dados claramente identificada. Antes de utilizá-los em análises, treinamento ou redistribuição, confirme a autorização do titular e as condições do provedor original. Este notebook não os baixa automaticamente.

In [ ]:
from google.colab import files

uploaded = files.upload()

saved_files = []
for name, content in uploaded.items():
    destination = RAW / Path(name).name
    destination.write_bytes(content)
    saved_files.append(destination)
    if destination.suffix.lower() == ".zip":
        with zipfile.ZipFile(destination) as zf:
            for member in zf.namelist():
                # Evita caminhos absolutos e path traversal.
                target_name = Path(member).name
                if not target_name or member.endswith("/"):
                    continue
                if Path(member).suffix.lower() == ".csv":
                    out = RAW / target_name
                    out.write_bytes(zf.read(member))
                    saved_files.append(out)

print("\nArquivos recebidos:")
for p in sorted(set(saved_files)):
    print("-", p.name, f"({p.stat().st_size:,} bytes)")

Saving b3essencial_stocks_d1.zip to b3essencial_stocks_d1.zip

Arquivos recebidos:
- 4GRO4L.csv (32 bytes)
- 4GTU4L.csv (71 bytes)
- A8BL4L.csv (32 bytes)
- A8CP7L.csv (71 bytes)
- A8GN8L.csv (72 bytes)
- A8GP7L.csv (32 bytes)
- A8SA5L.csv (32 bytes)
- A8SO4L.csv (32 bytes)
- A8TH4L.csv (32 bytes)
- A8VJ6L.csv (32 bytes)
- A8VJ7L.csv (32 bytes)
- A8VR5L.csv (73 bytes)
- A9GX5L.csv (32 bytes)
- A9MC6L.csv (72 bytes)
- A9XX4L.csv (32 bytes)
- AALR3.csv (2,344 bytes)
- ABCB10L.csv (73 bytes)
- ABCB4.csv (2,774 bytes)
- ABEV3.csv (2,875 bytes)
- ADMF3.csv (32 bytes)
- AERI3.csv (2,341 bytes)
- AFLT3.csv (1,728 bytes)
- AFST6L.csv (32 bytes)
- AG8B7L.csv (32 bytes)
- AGBT5L.csv (67 bytes)
- AGIR5L.csv (32 bytes)
- AGMO6L.csv (32 bytes)
- AGMO8L.csv (72 bytes)
- AGRN5L.csv (32 bytes)
- AGRN7L.csv (32 bytes)
- AGRO3.csv (2,632 bytes)
- AGSP5L.csv (32 bytes)
- AGXY3.csv (2,449 bytes)
- AHEB3.csv (155 bytes)
- AIPN5L.csv (32 bytes)
- AITP7L.csv (32 bytes)
- AITP8L.csv (70 bytes)
- ALLD3.csv (2,

## 3. Parser conservador de CSV

O parser abaixo reconhece alguns layouts comuns:
- M15 publicado no projeto `quant-research`: `Timestamp,Open,High,Low,Close,TickVolume,RealVolume,...`
- OHLC diário: `date,open,high,low,close,volume`
- CSV diário WIN antigo: `Data,Abertura,Max,Min,Fechamento`

O parser preserva OHLCV e não transforma volume de candle em agressão ou negócios individuais. O fuso horário permanece não verificado quando o arquivo não o declara.

In [ ]:
def _read_csv_flexible(path):
    errors = []
    for enc in ("utf-8-sig", "utf-8", "latin-1", "cp1252"):
        for sep in (",", ";"):
            try:
                df = pd.read_csv(path, encoding=enc, sep=sep)
                if len(df.columns) >= 4 and len(df) > 0:
                    df.columns = [str(c).strip() for c in df.columns]
                    return df, enc, sep
            except Exception as exc:
                errors.append(str(exc))
    raise ValueError(f"Não foi possível ler {Path(path).name}: {errors[-1:]}")


def normalize_ohlcv(path, instrument=None):
    raw, encoding, delimiter = _read_csv_flexible(path)
    lower = {c.lower().strip(): c for c in raw.columns}

    def pick(*names):
        for n in names:
            if n.lower() in lower:
                return lower[n.lower()]
        return None

    ts_col = pick("Timestamp", "datetime", "date", "Data")
    open_col = pick("Open", "open", "Abertura")
    high_col = pick("High", "high", "Max", "Máxima")
    low_col = pick("Low", "low", "Min", "Mínima")
    close_col = pick("Close", "close", "Fechamento")
    tickvol_col = pick("TickVolume", "tick_volume")
    realvol_col = pick("RealVolume", "real_volume")
    volume_col = pick("volume", "Volume", "VolumeFinanceiro", "qty")

    required = {"timestamp":ts_col, "open":open_col, "high":high_col,
                "low":low_col, "close":close_col}
    missing = [k for k,v in required.items() if v is None]
    if missing:
        raise ValueError(f"{Path(path).name}: colunas obrigatórias ausentes: {missing}")

    if ts_col.lower() == "data" and pick("Hora") is not None:
        timestamps = raw[ts_col].astype(str).str.strip() + " " + raw[pick("Hora")].astype(str).str.strip()
        dt = pd.to_datetime(timestamps, dayfirst=True, errors="coerce")
    else:
        dt = pd.to_datetime(raw[ts_col], errors="coerce", dayfirst=False)

    out = pd.DataFrame({
        "timestamp": dt,
        "open": pd.to_numeric(raw[open_col], errors="coerce"),
        "high": pd.to_numeric(raw[high_col], errors="coerce"),
        "low": pd.to_numeric(raw[low_col], errors="coerce"),
        "close": pd.to_numeric(raw[close_col], errors="coerce"),
    })

    if tickvol_col:
        out["tick_volume"] = pd.to_numeric(raw[tickvol_col], errors="coerce")
    if realvol_col:
        out["real_volume"] = pd.to_numeric(raw[realvol_col], errors="coerce")
    if volume_col and volume_col not in (tickvol_col, realvol_col):
        out["volume"] = pd.to_numeric(raw[volume_col], errors="coerce")

    inferred = instrument.upper() if instrument else None
    name = Path(path).name.upper()
    if inferred is None:
        if "WIN" in name: inferred = "WIN"
        elif "WDO" in name: inferred = "WDO"

    if inferred not in ("WIN", "WDO"):
        raise ValueError("Instrumento não identificado. Informe WIN ou WDO explicitamente.")

    out["instrument"] = inferred
    out["source_file"] = Path(path).name
    out["source_encoding"] = encoding
    out["source_delimiter"] = delimiter
    out["timezone_status"] = "UNVERIFIED"

    return out, raw.columns.tolist()

## 4. Auditoria de qualidade

A auditoria verifica datas inválidas, valores não numéricos, OHLC inconsistente, timestamps duplicados e ordenação. Ela não autentica a origem do arquivo nem confirma a metodologia de construção do contrato contínuo.

In [ ]:
def audit_ohlcv(df):
    issues = []
    n = len(df)

    if n == 0:
        return {"status":"INVALID", "rows":0, "issues":["Arquivo vazio"]}

    if df["timestamp"].isna().any():
        issues.append(f"Timestamps inválidos: {int(df['timestamp'].isna().sum())}")

    price_cols = ["open","high","low","close"]
    non_numeric = int(df[price_cols].isna().any(axis=1).sum())
    if non_numeric:
        issues.append(f"Linhas com preços ausentes/não numéricos: {non_numeric}")

    valid = df.dropna(subset=["timestamp"] + price_cols).copy()
    bad_ohlc = (
        (valid["low"] > valid[["open","close"]].min(axis=1)) |
        (valid["high"] < valid[["open","close"]].max(axis=1)) |
        (valid["high"] < valid["low"]) |
        (valid[price_cols] <= 0).any(axis=1)
    )
    if bad_ohlc.any():
        issues.append(f"Linhas com OHLC inválido: {int(bad_ohlc.sum())}")

    duplicate_ts = int(valid["timestamp"].duplicated().sum())
    if duplicate_ts:
        issues.append(f"Timestamps duplicados: {duplicate_ts}")

    sorted_ok = valid["timestamp"].is_monotonic_increasing
    if not sorted_ok:
        issues.append("Arquivo não está em ordem cronológica crescente")

    vol_cols = [c for c in ("tick_volume","real_volume","volume") if c in valid.columns]
    for col in vol_cols:
        bad = int((valid[col].dropna() < 0).sum())
        if bad:
            issues.append(f"{col}: {bad} valores negativos")

    return {
        "status": "VALID" if not issues else "INVALID_OR_WARNING",
        "instrument": str(df["instrument"].iloc[0]),
        "rows": n,
        "valid_rows": len(valid),
        "first_timestamp": str(valid["timestamp"].min()) if len(valid) else None,
        "last_timestamp": str(valid["timestamp"].max()) if len(valid) else None,
        "duplicate_timestamps": duplicate_ts,
        "chronological_ascending": bool(sorted_ok),
        "ohlc_invalid_rows": int(bad_ohlc.sum()) if len(valid) else None,
        "volume_columns": vol_cols,
        "aggression_available": False,
        "order_book_available": False,
        "provenance_verified": False,
        "issues": issues
    }

## 5. Descobrir e processar CSVs carregados

A identificação automática usa o nome do arquivo. Se o nome não indicar WIN ou WDO, será necessário informar o instrumento manualmente.

In [ ]:
csv_files = sorted(RAW.glob("*.csv"))
print("CSV encontrados:", len(csv_files))

datasets = {}
audit_reports = []

for path in csv_files:
    try:
        df, original_columns = normalize_ohlcv(path)
        report = audit_ohlcv(df)
        report["file"] = path.name
        report["original_columns"] = original_columns
        audit_reports.append(report)
        datasets[path.name] = df

        print("\n", path.name)
        print(json.dumps(report, ensure_ascii=False, indent=2))
    except Exception as exc:
        audit_reports.append({
            "file": path.name,
            "status": "NOT_PARSED",
            "issues": [str(exc)]
        })
        print("\nIGNORADO:", path.name, "|", exc)

audit_df = pd.DataFrame(audit_reports)
display(audit_df)

CSV encontrados: 617

IGNORADO: 4GRO4L.csv | Não foi possível ler 4GRO4L.csv: []

IGNORADO: 4GTU4L.csv | Instrumento não identificado. Informe WIN ou WDO explicitamente.

IGNORADO: A8BL4L.csv | Não foi possível ler A8BL4L.csv: []

IGNORADO: A8CP7L.csv | Instrumento não identificado. Informe WIN ou WDO explicitamente.

IGNORADO: A8GN8L.csv | Instrumento não identificado. Informe WIN ou WDO explicitamente.

IGNORADO: A8GP7L.csv | Não foi possível ler A8GP7L.csv: []

IGNORADO: A8SA5L.csv | Não foi possível ler A8SA5L.csv: []

IGNORADO: A8SO4L.csv | Não foi possível ler A8SO4L.csv: []

IGNORADO: A8TH4L.csv | Não foi possível ler A8TH4L.csv: []

IGNORADO: A8VJ6L.csv | Não foi possível ler A8VJ6L.csv: []

IGNORADO: A8VJ7L.csv | Não foi possível ler A8VJ7L.csv: []

IGNORADO: A8VR5L.csv | Instrumento não identificado. Informe WIN ou WDO explicitamente.

IGNORADO: A9GX5L.csv | Não foi possível ler A9GX5L.csv: []

IGNORADO: A9MC6L.csv | Instrumento não identificado. Informe WIN ou WDO explicitam

,file,status,issues
0,4GRO4L.csv,NOT_PARSED,[Não foi possível ler 4GRO4L.csv: []]
1,4GTU4L.csv,NOT_PARSED,[Instrumento não identificado. Informe WIN ou ...
2,A8BL4L.csv,NOT_PARSED,[Não foi possível ler A8BL4L.csv: []]
3,A8CP7L.csv,NOT_PARSED,[Instrumento não identificado. Informe WIN ou ...
4,A8GN8L.csv,NOT_PARSED,[Instrumento não identificado. Informe WIN ou ...
...,...,...,...
612,WLMM3.csv,NOT_PARSED,[Instrumento não identificado. Informe WIN ou ...
613,WLMM4.csv,NOT_PARSED,[Instrumento não identificado. Informe WIN ou ...
614,WTXA11.csv,NOT_PARSED,[Não foi possível ler WTXA11.csv: []]
615,YDUQ3.csv,NOT_PARSED,[Instrumento não identificado. Informe WIN ou ...


## 6. Preparar datasets válidos para pesquisa

Somente arquivos aprovados na auditoria estrutural serão gravados na pasta `data/audited`. Os dados continuam marcados como procedência não verificada. Não há inferência de agressão, book ou trades a partir dos candles.

In [ ]:
approved = []

for filename, df in datasets.items():
    report = next((r for r in audit_reports if r.get("file") == filename), {})
    if report.get("status") == "VALID":
        clean = df.sort_values("timestamp").reset_index(drop=True)
        out_path = AUDITED / f"{Path(filename).stem}_AUDITED.csv"
        clean.to_csv(out_path, index=False)
        approved.append((filename, clean, out_path))

print("Arquivos aprovados estruturalmente:", len(approved))
for filename, df, out_path in approved:
    print(filename, "->", out_path.name, "|", len(df), "linhas")

Arquivos aprovados estruturalmente: 0


## 7. Visualização exploratória opcional

A visualização abaixo mostra somente o fechamento dos candles. Não constitui sinal de compra ou venda.

In [ ]:
for filename, df, _ in approved:
    plt.figure(figsize=(12, 4))
    plt.plot(df["timestamp"], df["close"], linewidth=0.8)
    plt.title(f"{filename} — fechamento histórico (PAPER_ONLY)")
    plt.xlabel("Timestamp")
    plt.ylabel("Preço")
    plt.grid(True, alpha=0.25)
    plt.tight_layout()
    plt.show()

## 8. Exportar relatórios de auditoria

Os relatórios e CSVs normalizados podem ser baixados para revisão. Não compartilhe dados licenciados sem verificar os termos de uso.

In [ ]:
audit_json = REPORTS / "dataset_audit_report.json"
audit_csv = REPORTS / "dataset_audit_report.csv"

audit_json.write_text(
    json.dumps(audit_reports, ensure_ascii=False, indent=2),
    encoding="utf-8"
)
audit_df.to_csv(audit_csv, index=False)

print("Relatórios criados:")
print(audit_json)
print(audit_csv)

# Para baixar pelo navegador do Colab, descomente:
# from google.colab import files
# files.download(str(audit_json))
# files.download(str(audit_csv))

print("\nMODO FINAL:", MODE)
print("LIVE_TRADING:", LIVE_TRADING)
print("ORDER_ROUTING:", ORDER_ROUTING)

Relatórios criados:
/content/ai_trading_agent/reports/dataset_audit_report.json
/content/ai_trading_agent/reports/dataset_audit_report.csv

MODO FINAL: PAPER_ONLY
LIVE_TRADING: False
ORDER_ROUTING: False
